# 00 · Utilities

Shared configuration and helpers for the John Doe Foods pipeline. The other notebooks load this one with `%run 00_utils`.

All data is fictional. It comes from [demo-data-generator](https://github.com/lodeanalytics/demo-data-generator).

In [ ]:
import time
import uuid
from datetime import date, datetime, timedelta, timezone

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

## Configuration

In [ ]:
# Source files land here, one subfolder per delivery, e.g. Files/landing/john_doe_foods/2026-10-31/
LANDING_ROOT = "Files/landing/john_doe_foods"

# Entities the pipeline knows about. A CSV whose name isn't listed is skipped and reported.
ENTITIES = [
    "products",
    "customers",
    "sales_order_headers",
    "sales_order_lines",
    "inventory_snapshots",
    "production_runs",
    "downtime_events",
]

# Weather enrichment: one city per sales region, pulled from the Open-Meteo archive API.
WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_START = "2025-01-01"   # first day to backfill
WEATHER_LAG_DAYS = 5           # the archive runs a few days behind today
REGION_CITIES = {
    # region: (city, latitude, longitude)
    "North":   ("Minneapolis",  44.98,  -93.27),
    "South":   ("Dallas",       32.78,  -96.80),
    "East":    ("Philadelphia", 39.95,  -75.17),
    "West":    ("Denver",       39.74, -104.99),
    "Central": ("Kansas City",  39.10,  -94.58),
}

## Helpers

In [ ]:
def log(message):
    print(f"[{datetime.now():%H:%M:%S}] {message}")


def new_batch_id():
    """A sortable, unique id for one notebook run."""
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]


def table_exists(name):
    return spark.catalog.tableExists(name)


def list_files(root, suffix=".csv"):
    """Every file under root (recursively) whose name ends with suffix, sorted by path."""
    found = []

    def walk(path):
        for item in notebookutils.fs.ls(path):
            if item.isDir:
                walk(item.path)
            elif item.name.lower().endswith(suffix):
                found.append(item.path)

    walk(root)
    return sorted(found)


def append(df, table):
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table)


def overwrite(df, table):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)


def upsert(df, table, keys):
    """Insert new rows and update existing ones, matched on the key columns (Delta MERGE)."""
    if not table_exists(table):
        overwrite(df, table)
        return
    from delta.tables import DeltaTable

    condition = " AND ".join(f"t.`{k}` = s.`{k}`" for k in keys)
    (
        DeltaTable.forName(spark, table).alias("t")
        .merge(df.alias("s"), condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )